In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
mask_metadata = pd.read_csv("../metadata/Recording_metadata.csv")
mask_metadata

,Brain,Recording,Stimulus,Mask,Plane,Region,Side,Imaging_FPS,Video_FPS,Feeding_start_time (s),Feeding_stop_time (s),Feeding_duration (s)
0,AKG_B30,R1,NHS60,5,1,gamma1pedc,Right,1.254170,15.0,217.066667,519.866667,302.800000
1,AKG_B30,R1,NHS60,2,1,gamma1pedc,Left,1.254170,15.0,217.066667,519.866667,302.800000
2,AKG_B31,R1,NHS60,2,0,gamma1pedc,Left,1.377122,15.0,218.533333,522.733333,304.200000
3,AKG_B31,R1,NHS60,3,1,gamma1pedc,Right,1.377088,15.0,218.533333,522.733333,304.200000
4,AKG_B32,R1,NHS60,2,0,gamma1pedc,Left,1.251235,15.0,222.266667,516.400000,294.133333
5,AKG_B32,R1,NHS60,3,1,gamma1pedc,Left,1.251212,15.0,222.266667,516.400000,294.133333
6,AKG_B32,R1,NHS60,6,1,gamma1pedc,Right,1.251212,15.0,222.266667,516.400000,294.133333
7,AKG_B33,R1,NHS60,6,0,gamma1pedc,Right,1.281426,15.0,233.600000,534.800000,301.200000
8,AKG_B33,R1,NHS60,2,0,gamma1pedc,Left,1.281426,15.0,233.600000,534.800000,301.200000
9,AKG_B34,R1,NHS60,2,0,gamma1pedc,Left,1.316616,15.0,218.866667,519.133333,300.266667


In [3]:
stimuli = mask_metadata["Stimulus"].unique()

region = "gamma1pedc"
    
for stimulus in stimuli:

    df_data = pd.read_csv(f"../csv_files/Brainwise_aligned_dFF_traces_of_{region}_for_{stimulus}.csv",index_col=0)
    
    feeding_data_summary = []
    
    for brain in df_data.columns:
    
        feeding_1_start_time = 0
        feeding_1_duration = mask_metadata[(mask_metadata["Region"] == region) & (mask_metadata["Brain"] == brain)]["Feeding_duration (s)"].unique()[0]
        feeding_1_stop_time = feeding_1_start_time + feeding_1_duration
        
        pre_feeding_idx = df_data[df_data.index < feeding_1_start_time*1000].index
        during_feeding_idx = df_data[(df_data.index >= feeding_1_start_time*1000) & (df_data.index < feeding_1_stop_time*1000)].index
        post_feeding_window1_idx = df_data[(df_data.index >= feeding_1_stop_time*1000) & (df_data.index < (feeding_1_stop_time + 300)*1000)].index
        post_feeding_window2_idx = df_data[(df_data.index >= (feeding_1_stop_time + 300)*1000) & (df_data.index < (feeding_1_stop_time + 600)*1000)].index
        post_feeding_window3_idx = df_data[(df_data.index >= (feeding_1_stop_time + 600)*1000) & (df_data.index < (feeding_1_stop_time + 900)*1000)].index
        post_feeding_window4_idx = df_data[(df_data.index >= (feeding_1_stop_time + 900)*1000) & (df_data.index < (feeding_1_stop_time + 1200)*1000)].index
        post_feeding_window5_idx = df_data[(df_data.index >= (feeding_1_stop_time + 1200)*1000) & (df_data.index < (feeding_1_stop_time + 1500)*1000)].index
        
        dFF_trace_brain = df_data[brain]
        
        pre_feeding_mean = np.nanmean(dFF_trace_brain[pre_feeding_idx])
        during_feeding_mean = np.nanmean(dFF_trace_brain[during_feeding_idx])
        post_feeding_window1_mean = np.nanmean(dFF_trace_brain[post_feeding_window1_idx])
        post_feeding_window2_mean = np.nanmean(dFF_trace_brain[post_feeding_window2_idx])
        post_feeding_window3_mean = np.nanmean(dFF_trace_brain[post_feeding_window3_idx])
        post_feeding_window4_mean = np.nanmean(dFF_trace_brain[post_feeding_window4_idx])
        post_feeding_window5_mean = np.nanmean(dFF_trace_brain[post_feeding_window5_idx])
        
        feeding_data_summary.append([brain,pre_feeding_mean,during_feeding_mean,post_feeding_window1_mean,post_feeding_window2_mean,post_feeding_window3_mean,post_feeding_window4_mean,post_feeding_window5_mean])
    
    df_feeding_data_summary_stimulus = pd.DataFrame(np.array(feeding_data_summary),columns=["BrainID","PreMean","DuringMean","Post1Mean","Post2Mean","Post3Mean","Post4Mean","Post5Mean"])
    df_feeding_data_summary_stimulus.to_csv(f"../csv_files/Brainwise_time_binned_mean_dFF_response_of_{region}_for_{stimulus}.csv",index=None)
    